# Current-video full diarization and additive overlap extraction

Attach a Kaggle dataset containing the selected YouTube video named `<video-id>_full480.mp4` or `<video-id>.mp4`. An additive Sortformer policy is optional for a new-video baseline run. The notebook preserves the evidence-based baseline; supplemental stages never overwrite it.


In [ ]:
from pathlib import Path
import subprocess, sys, os, json, shutil, time, zipfile
from urllib.parse import urlparse, parse_qs

VIDEO_URL = 'https://www.youtube.com/watch?v=VvPpdeqwHBs'
NOTEBOOK_REVISION = 'resumable-window-checkpoints-v32'
REQUIRE_OVERLAP_POLICY = False
RUN_FULL_VIDEO = True
RUN_TARGETED_REVIEW = True
RUN_OVERLAP_EXTRACTION = True
RUN_MOSSFORMER2_REVIEW = True
RUN_CAPTION_GAP_REVIEW = True
RUN_DIAPER_OVERLAP = True
HAS_OPENING_REFERENCE = False
REVIEW_WEAK_CONFIDENCE = 0.35
REVIEW_SHORT_SECONDS = 1.0
BATCH_SIZE = 4

parsed_video_url = urlparse(VIDEO_URL)
VIDEO_ID = (parsed_video_url.path.strip('/') if parsed_video_url.netloc == 'youtu.be'
            else parse_qs(parsed_video_url.query).get('v', [''])[0])
if not VIDEO_ID or any(character not in 'abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789_-' for character in VIDEO_ID):
    raise ValueError(f'Could not derive a safe YouTube video ID from {VIDEO_URL!r}')

ON_KAGGLE = Path('/kaggle/input').exists()
if ON_KAGGLE:
    probe = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True) if shutil.which('nvidia-smi') else None
    if probe is None or probe.returncode or 'GPU ' not in probe.stdout:
        raise RuntimeError('Enable a GPU accelerator in Kaggle Settings, then rerun this cell.')
    print(probe.stdout)
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN_VALUE = UserSecretsClient().get_secret('HF_TOKEN')
    if not HF_TOKEN_VALUE:
        raise RuntimeError('Add and enable the private Kaggle secret HF_TOKEN, then rerun.')
    print('Hugging Face credentials configured; token not displayed.')
    BASE = Path('/kaggle/working')
else:
    HF_TOKEN_VALUE = None
    BASE = Path.cwd()/'diarization-run'
BASE.mkdir(parents=True, exist_ok=True)
WORK = BASE/'diarization'; WORK.mkdir(exist_ok=True)
RESULTS = BASE/('results-'+VIDEO_ID); RESULTS.mkdir(exist_ok=True)
CACHE = BASE/'stage-cache'/VIDEO_ID
VENV = BASE/'diarization-venv'
PYTHON = str(VENV/('Scripts/python.exe' if os.name == 'nt' else 'bin/python'))
VIDEO = WORK/('video-'+VIDEO_ID+'-h264.mp4')
REFERENCE = WORK/'target-reference'; REFERENCE.mkdir(exist_ok=True)
print('Video URL:', VIDEO_URL)
print('Video ID:', VIDEO_ID)
print('Notebook revision:', NOTEBOOK_REVISION)
print('Results folder:', RESULTS)
print('Setup will use:', PYTHON)


## Install and verify

Enable Internet and a GPU before running. The setup uses an isolated environment and verifies CUDA before the full video starts.


In [ ]:
import base64
import zlib
SOURCE_ARCHIVE = '...'
EMBEDDED_FILES = json.loads(zlib.decompress(base64.b64decode(SOURCE_ARCHIVE)))
for name, source in EMBEDDED_FILES.items():
    (WORK/name).write_text(source)

ENV = os.environ.copy()
ENV['PYTHONUNBUFFERED'] = '1'
ENV['MPLCONFIGDIR'] = str(BASE/'matplotlib-cache')
ENV['MPLBACKEND'] = 'Agg'
ENV['NUMBA_CACHE_DIR'] = str(BASE/'numba-cache')
if HF_TOKEN_VALUE:
    ENV['HF_TOKEN'] = HF_TOKEN_VALUE
    ENV['HUGGING_FACE_HUB_TOKEN'] = HF_TOKEN_VALUE
def checked(command, **kwargs):
    return subprocess.run(command, env=ENV, check=True, **kwargs)

print('1/4: Creating/checking the isolated environment', flush=True)
ready = False
if Path(PYTHON).exists():
    ready = subprocess.run([PYTHON, '-m', 'pip', '--version'], env=ENV,
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL).returncode == 0
if not ready:
    bootstrap = BASE/'bootstrap-tools'
    checked([sys.executable, '-m', 'pip', 'install', '--target', str(bootstrap),
             'virtualenv>=20.26,<21', 'wrapt'])
    bootstrap_env = ENV.copy(); bootstrap_env['PYTHONPATH'] = str(bootstrap)
    subprocess.run([sys.executable, '-m', 'virtualenv', '--no-download', str(VENV)],
                   env=bootstrap_env, check=True)

print('2/4: Installing the pipeline and extraction model', flush=True)
requirements = [
    'whisperx==3.8.6', 'speechbrain==1.1.1', 'insightface==2.0',
    'torch==2.8.0', 'torchaudio==2.8.0', 'numpy==2.5.3',
    'opencv-python==5.0.0.93', 'onnxruntime-gpu==1.23.2', 'wrapt',
    'yt-dlp', 'soundfile', 'safe-gpu', 'yamlargparse==1.31.1',
    'decorator', 'h5py', 'matplotlib', 'librosa', 'scikit-learn', 'tensorboard',
]
checked([PYTHON, '-m', 'pip', 'install', '--upgrade', 'pip'])
checked([PYTHON, '-m', 'pip', 'install', *requirements])
checked([PYTHON, '-m', 'pip', 'install', '--no-deps', 'clearvoice==0.1.2'])
checked([PYTHON, '-m', 'pip', 'install', 'gdown', 'librosa==0.10.2.post1',
         'rotary-embedding-torch==0.8.3', 'scenedetect==0.6.6',
         'python-speech-features==0.6', 'torchinfo', 'pydub'])
checked([PYTHON, '-m', 'pip', 'install',
         'git+https://github.com/wenet-e2e/wespeaker.git'])
# WeSep's current package metadata omits its namespace-style wesep/utils
# directory. Keep the checkout and put it first on PYTHONPATH so the complete
# source tree is used, while pip still installs all declared dependencies.
WESEP_SOURCE = WORK/'vendor'/'wesep'
WESEP_REVISION = '99eca54b60300d39b9353d93cf285a14bba37854'
wesep_revision_file = WESEP_SOURCE/'.codex-compatible-revision'
if (not (WESEP_SOURCE/'wesep'/'utils'/'utils.py').is_file()
        or not wesep_revision_file.is_file()
        or wesep_revision_file.read_text().strip() != WESEP_REVISION):
    if WESEP_SOURCE.exists():
        shutil.rmtree(WESEP_SOURCE)
    WESEP_SOURCE.parent.mkdir(parents=True, exist_ok=True)
    checked(['git', 'clone', '--no-checkout',
             'https://github.com/wenet-e2e/wesep.git', str(WESEP_SOURCE)])
    checked(['git', '-C', str(WESEP_SOURCE), 'checkout', WESEP_REVISION])
    wesep_revision_file.write_text(WESEP_REVISION + '\n')
checked([PYTHON, '-m', 'pip', 'install', str(WESEP_SOURCE)])
# The upstream wheel omits wesep/utils because that directory has no
# __init__.py. Overlay the complete checkout onto site-packages so imports do
# not depend on PYTHONPATH or notebook process state.
site_packages = Path(subprocess.check_output(
    [PYTHON, '-c', 'import site; print(site.getsitepackages()[0])'],
    env=ENV, text=True).strip())
installed_wesep = site_packages/'wesep'
shutil.copytree(WESEP_SOURCE/'wesep', installed_wesep, dirs_exist_ok=True)
missing_utility = installed_wesep/'utils'/'utils.py'
if not missing_utility.is_file():
    raise RuntimeError(f'WeSep repair failed; missing {missing_utility}')
# These upstream source directories contain Python modules but omit package
# markers, which is also why they disappear from the built wheel.
for directory in [installed_wesep/'utils', installed_wesep/'dataset', installed_wesep/'bin']:
    if directory.is_dir():
        (directory/'__init__.py').touch()
ENV['PYTHONPATH'] = str(WORK) + os.pathsep + ENV.get('PYTHONPATH', '')

print('3/4: Selecting the CUDA ONNX runtime', flush=True)
checked([PYTHON, '-m', 'pip', 'uninstall', '-y', 'onnxruntime', 'onnxruntime-gpu'])
checked([PYTHON, '-m', 'pip', 'install', '--no-deps', '--force-reinstall',
         'onnxruntime-gpu==1.23.2'])
if shutil.which('ffmpeg') is None:
    raise RuntimeError('ffmpeg is required. Kaggle normally includes it.')
library_dirs = subprocess.check_output([PYTHON, '-c',
    "import site,pathlib; print(':'.join(str(p) for d in site.getsitepackages() for p in pathlib.Path(d).glob('nvidia/*/lib')))"], env=ENV, text=True).strip()
ENV['LD_LIBRARY_PATH'] = library_dirs + ':' + ENV.get('LD_LIBRARY_PATH', '')

print('4/4: Verifying GPU imports and focused behavior tests', flush=True)
verification = """import torch,onnxruntime as ort,wrapt,wesep
import chainofrules,repeat_evidence
from wesep.models import get_model
print('Torch:', torch.__version__, 'CUDA build:', torch.version.cuda)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE (local CPU)')
print('ONNX providers:', ort.get_available_providers())
print('WeSep repaired package:', wesep.__file__)
assert get_model('BSRNN').__name__ == 'BSRNN', 'WeSep English checkpoint is incompatible'
"""
if ON_KAGGLE:
    verification += "assert torch.cuda.is_available(), 'Kaggle GPU is unavailable'\nassert 'CUDAExecutionProvider' in ort.get_available_providers(), 'GPU ONNX runtime is unavailable'\n"
checked([PYTHON, '-c', verification], cwd=WORK)
checked([PYTHON, '-m', 'unittest', 'test_cloud_runtime', 'test_short_answers',
         'test_repeat_evidence', 'test_overlap_resolution',
         'test_overlap_extraction_review', 'test_confident_transcript',
         'test_mossformer2_review_policy',
         'test_reference_promotion', 'test_diaper_overlap'], cwd=WORK)

import hashlib
REFERENCE_FILES = {'face_embeddings.npy': '...',
 'reference.json': 'ewogICJtZXRob2QiOiAibWFudWFsIGlkZW50aXR5IGFwcHJvdmFsIGZvbGxvd2VkIGJ5IGNvbnNpc3RlbmN5IHNjcmVlbmluZyIsCiAgInNjcmVlbmluZyI6IHsKICAgICJ2b2ljZSI6IHsKICAgICAgImFuY2hvcl9yb3ciOiAwLAogICAgICAiYWNjZXB0ZWRfcm93cyI6IFsKICAgICAgICAwLAogICAgICAgIDEsCiAgICAgICAgMywKICAgICAgICA0LAogICAgICAgIDUsCiAgICAgICAgNiwKICAgICAgICA3LAogICAgICAgIDgKICAgICAgXSwKICAgICAgImV4Y2x1ZGVkX3Jvd3MiOiBbCiAgICAgICAgMgogICAgICBdLAogICAgICAic2ltaWxhcml0eV90b19hbmNob3IiOiB7CiAgICAgICAgIjAiOiAxLjAwMDAwMDExOTIwOTI4OTYsCiAgICAgICAgIjEiOiAwLjY1MTYzNjgzODkxMjk2MzksCiAgICAgICAgIjIiOiAwLjQwNjA5NTYyMzk3MDAzMTc0LAogICAgICAgICIzIjogMC41MDI1NDQyMjQyNjIyMzc1LAogICAgICAgICI0IjogMC40OTkxMTY1OTk1NTk3ODM5NCwKICAgICAgICAiNSI6IDAuNTk0MzQ5NjgyMzMxMDg1MiwKICAgICAgICAiNiI6IDAuNTI3MDIzMjU1ODI1MDQyNywKICAgICAgICAiNyI6IDAuNTEyNTQ5NDAwMzI5NTg5OCwKICAgICAgICAiOCI6IDAuNjI3NzY3NzQxNjgwMTQ1MwogICAgICB9LAogICAgICAibWluaW11bV9zaW1pbGFyaXR5IjogMC40NQogICAgfSwKICAgICJmYWNlIjogewogICAgICAiYW5jaG9yX3JvdyI6IDMsCiAgICAgICJhY2NlcHRlZF9yb3dzIjogWwogICAgICAgIDAsCiAgICAgICAgMSwKICAgICAgICAyLAogICAgICAgIDMsCiAgICAgICAgNCwKICAgICAgICA1LAogICAgICAgIDYsCiAgICAgICAgNywKICAgICAgICA4LAogICAgICAgIDksCiAgICAgICAgMTAsCiAgICAgICAgMTEKICAgICAgXSwKICAgICAgImV4Y2x1ZGVkX3Jvd3MiOiBbXSwKICAgICAgInNpbWlsYXJpdHlfdG9fYW5jaG9yIjogewogICAgICAgICIwIjogMC40Njg3OTQ0NjUwNjUwMDI0NCwKICAgICAgICAiMSI6IDAuNjYzMTgyMzc3ODE1MjQ2NiwKICAgICAgICAiMiI6IDAuODY4NjU2NTE2MDc1MTM0MywKICAgICAgICAiMyI6IDEuMCwKICAgICAgICAiNCI6IDAuODcwMjY0NzY4NjAwNDYzOSwKICAgICAgICAiNSI6IDAuNzgzMjY3MTk5OTkzMTMzNSwKICAgICAgICAiNiI6IDAuODczNTI0MDEwMTgxNDI3LAogICAgICAgICI3IjogMC45MTgzMjc4MDgzODAxMjcsCiAgICAgICAgIjgiOiAwLjg1NDcxODU2NTk0MDg1NjksCiAgICAgICAgIjkiOiAwLjg5MDk0MDk2NDIyMTk1NDMsCiAgICAgICAgIjEwIjogMC44NDQxOTExMzM5NzU5ODI3LAogICAgICAgICIxMSI6IDAuODcyMTU4NTI3Mzc0MjY3NgogICAgICB9LAogICAgICAibWluaW11bV9zaW1pbGFyaXR5IjogMC40NQogICAgfQogIH0sCiAgInZvaWNlX3NvdXJjZXMiOiBbCiAgICAiODNlMGU4ZTU2NzA3NGI1NWE1MDJiOTMxNzIyMDI5NTkiLAogICAgImNkNmMyZjc1NjBhMjRlYjlhYmRkNjVkNGI4N2MyNGNlIiwKICAgICJkOTEyNDJlZTM3MWI0NDQ0YWUxYmE1N2YyZjYwNGQ2ZCIsCiAgICAiZGNiODFjZjFmNjU1NGIzZjhhMDc4MTk3MDU2ZTQ3MDYiLAogICAgIjYzNjZhNmI4ODg2NDQzNTRhOWUyYmM3M2JjZGQxZTM3IiwKICAgICJjNTQxNDM1NTJlY2Q0Njk5OTc0N2QzMTUwZTQ3NjI3OSIsCiAgICAiYmYwNDhiMGI4NjYyNGQzNWIxZmExY2I4ODFiZGI1MmUiLAogICAgIjJiZDRhNTljMzAzMDQyN2U5ZTgxNWYzYWQwMTgyMjQ5IiwKICAgICI2ZjMxNTlhOThkYjY0OTljYjdjZWZlOGU2OGVhYjU2YSIKICBdLAogICJmYWNlX3NvdXJjZXMiOiBbCiAgICAiMjIyYTQ4MjM4MDVmNDQ0ZWI5Y2ZiZDYzZDFkM2QwZDIiLAogICAgImJjOTE1MGYyNzI4MjQzYjE5N2E4ZTAwNWUzYmYxMGI2IiwKICAgICI4M2UwZThlNTY3MDc0YjU1YTUwMmI5MzE3MjIwMjk1OSIsCiAgICAiY2Q2YzJmNzU2MGEyNGViOWFiZGQ2NWQ0Yjg3YzI0Y2UiLAogICAgImQ5MTI0MmVlMzcxYjQ0NDRhZTFiYTU3ZjJmNjA0ZDZkIiwKICAgICJkY2I4MWNmMWY2NTU0YjNmOGEwNzgxOTcwNTZlNDcwNiIsCiAgICAiNjM2NmE2Yjg4ODY0NDM1NGE5ZTJiYzczYmNkZDFlMzciLAogICAgImM1NDE0MzU1MmVjZDQ2OTk5NzQ3ZDMxNTBlNDc2Mjc5IiwKICAgICIzYWFiMmU3ZjQwOWM0OThkYTU3MmNlZDRjMDI1YmEwYSIsCiAgICAiYmYwNDhiMGI4NjYyNGQzNWIxZmExY2I4ODFiZGI1MmUiLAogICAgIjJiZDRhNTljMzAzMDQyN2U5ZTgxNWYzYWQwMTgyMjQ5IiwKICAgICI2ZjMxNTlhOThkYjY0OTljYjdjZWZlOGU2OGVhYjU2YSIKICBdLAogICJzaG9ydF9saWJyYXJ5X2lkcyI6IFtdLAogICJub3RlIjogIk5vIGluZmVyZW5jZSBpcyBhbiBhcHByb3ZhbC4gU2hvcnQgc2FtcGxlcyBleGNsdWRlZCBmcm9tIHRoZSBtYWluIHZvaWNlIGNlbnRyb2lkLiBIb2xkIGV2YWx1YXRpb24gdmlkZW9zIG91dCBvZiBlbnJvbGxtZW50LiIKfQo=',
 'voice_embeddings.npy': 'k05VTVBZAQB2AHsnZGVzY3InOiAnPGY0JywgJ2ZvcnRyYW5fb3JkZXInOiBGYWxzZSwgJ3NoYXBlJzogKDgsIDE5MiksIH0gICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIArK5nM9Y0sOPY9lJD5Ldyy9EBwjvRHPHr6HjJu9gS/UPQOQrjqJtQq9Sb7WvfAp4z1ivZk87685PaoFmL35zlu9JzloPerd6b1ChyQ9IZO7PWzyOD2uIYO9KcMFvl/3eL1K5o09h5SHPUhflL2prMS998TrPZ2/R7xcy4U9Oqv9vGkYdL0Hrka8TnfoPWd9tL3Ctue9O8KXPffqqbxvh8U8JHG5u2s4iL1/gRK+XFS6PTWrlj1Dn509xhXavdVccryB28O7ocaAvS/Su7yrFfE9bE3gvdck1D1inNA9PIvpOxKsSbxF/vk9UoYTvc8v27tLI1u8TBqAvJjMyz02ctk9Z0cfvX/Tzr0LRTk904gLveSLtzzhT0m9+qQDvQXPCz1IfEK9/OSmvI/KCz5amum92I3OuyiyNr24t3g9ujMCvbOqjj2/92U8osiqPW+j872FERI96CLZPa8boLvd25M9xmqlO8RXsTqs+Ou8+CDkPUwZNT0rpvA9zWaEvdr3yD2WE5w7w0J4vf+25DpANjM9YniBPLyf2T2/cBG+eWvIvWSL2Tzgz6i97TVcPVENvLwsi9g886nQvNLpgT0Cq5u92HwwveUGX72idT6936gove0Ug7y2ccK8BXyFPfJom718u5M9zZTevBTtQjwbYqE8OwCHvbqQbTzkyuq832w4vvG2wTsoheI93zeuuQ5AXT0368G8H8Ievm95oD2MF9C8Omp2PcXeGj7OW0m9hGZvvdCT7bxd9ti9/+u/vVX3Kr11PfI8uPOPPeOORD7IIsA8C9j0vermBT5FyrC8UV/svEO3obwkv0c9YMTLPEE01r3xka08vwXIu0zdBj2+5Ze9+yvFPZKdtDz27ci7xObRPQtbFb1Z4wu9F16yPZeQaDzv9Fy9meoPvfU4xTz9gUs97XhivQPyZ737a4C7axDnPbktN71Sipq8PbnrPBV/DT7djNw9DHIyvXZ5Hr1qWIm8A8yMvF8Cg7wMFKu9E/pavbeqij0OE3I9W2DevT9MJz3Yt4E5mpCnPI+qQT1Q8G+94UCWPKYnzb24aNm90PCyPZsNIj3mvYy9Gak+PeeeEj0mb0Y+/6vQvGVcor0c7646lZ2MO6B6uL2PVYI8wrliPJHMqD2qF12952azvdZtVL0wCZW6oZr6PYmwj707cby9KLvWPY0WrD0kOZ09iesVPX66Db7rqmG7gtDFPaMmFL1z/cK887g0Pf9ojbzUeX89V9ZUvetszb3BBiC8R7e4vAfOgT2abvk7QltJPGNiKL3klps99xAjvJFQUbziMIY9CzuovUr2azykHMQ9zrCrvPjfk70l8XM8zvjzvOnxz7x9F1q9kLfaO/uqYb3s31o8quSAvbefSL3KQh26ntnuvEo0grxorM68Sim3vbPdbD1AzTA9GXBEPIKVtjwqyOG9UaAxPSQ9tzysrMq8gw8IvSmSGz6/jF49L3NhPW0fNL6HDoc8dr6uPe0uCzwryrY8vuXfvMeUpT1xdxm+fTL/PYqwOr1MJb09t6TxvXomlj0aaUa8pBA6vCHIGL11FYA9It88PPIeAT1VXkW+XccPPNKKK72r7Kw8IXBAPZUNYjzfdBa9qQ2gu/FtHD63khE9tIgNvbXOWz0ILrG9GEqnvUePAL3qiTs9/i9TPb1pHb2r4jM8zpXYuxRpZTvniEo7X5Nxvaz44LeBcQe9GXLOvQjgg71ERjg+uwHZvckQKD1qnJQ8lF6YvVv3Bj6ULya96w/cO9dR+j26wce8bxCEvTeMCr0AVIm9FvjUvaeTML2MdKS9K7TjPXBibT4Kgpe8o685vm+zQD4rE2O94JCovLL2S724TUe72qejvIitdLxhEE49y2KNPOPOlLzgf3G9RPhAPdgiLj0L/GC9Al6ZPe+TCb4IVVi9YhLFPdY3+7wSCQu+1BoMvlw12TxE22y9orUNvjwnlTz2a9O8Gj8DPOB5xr3NhBc8bJN+vfRQxjy//gM+N+kUvIuHWjyAQ9C9ibpBPexdSz0hkly8hVyjvdGlTD1PWls9dyIVPdobkT1eqLm9isO8Pe4hbj4U0oC9D2a0PDKNyrzLW128WwPKPWd+TD0tLmI811+ePcpPRbzzLeY8lzCGPaDIML4FeBU9+JzmvG81gb2hbvG8t5sYPB0lM73XNDO9od/9vdl+FL3Hth69CuLnPOFYjry/2Qa93qy1vTdXHT0NRlo98cMevJtBBLskN2u97wNYPWqTir0XPtu9qZ6uuxLnHL68sUi8vmIKO0Z8Ur2OfAO9MrefvYjQJjxHRXQ9yriKvHntAD1FOrq91kB1vddspb1ITSY+2KyVvdnjfjw5RBs9NCMXvYGte72RfQC99goWvYE6ZzzFXme9QSRBvU3lDr2LRdM9nZTjPL+jZLw9xYs9FcTyvaSo6ju4ISW9HusvPQVopD1B5zo9ifURvTUXvj0Zq8u99Y+pPTOKaz2DnaI868O6vVJ+HT58XD27CwsbPjA3zr19DU0++zukPVElRD3+JAU+VdMavvye871Q1tG91qI0OQrLSb2P/8M96/+vvdVekD2tJSQ8w+qqvY1lDT0vFd096hKEO1AVITuKVNq9AVmyvUW+pLxvZSe++MG+PAgQjT2VdXo9m2etPC5lET1hbIa5hEPAvdKylrzpeuO8Skr2uqbcRb0lLsm9e3Fquz7ftb1xMsa9dkJOPdfozTvcfDO9goLTvQzg9bxjZlM9ABiVvYm1ID1dfia70JB0vMZt3zyI+jS9KqmjvS0Cyz05L7A9gi3HPXa9gT2QlgS74uyCvYpD/rw1rji9NvKDvQqSl7yT0LC9oWSvPXGVCj4u7Pm8z4o0PbbcKT3AQo29WM2ZPOYfk7yj05a9ohlGvWWetL2YLs68Av8HPs26Qj0G47C7KcqIvRPbcz37BSu9wBIFPvS9l71yoEQ8gs6du5zxwryhP9u8xZLQvc68T7yURpM8ktwavg/tI71Swca9C86uPeKgMb2gmym8v39uvXjrtj36JgI+iCWhPTPPm718fXa9cRi4vGkDkD0WvJy95jnBPJxuZjteDce9H65Mu6jNDT0g9ki9MytXvAH/sD0+w8O8NYBsvXXAFrsE/AK+o5J3PdON4byoBAQ8AyyAPXV4Iz3YNhw+oJ3XPQ5aOb7RL807/m4bvSSop72emzy9/9GSPAyB3D11xrc8Xb8Dvt3IiD1uZZO8ASsuPSa9wb2EHLC9EogOvFxxqr1jmcG8yWThvWPMyDp1h0k94QzcPGGrWD2V7+S9A7Wauxb7Sb4l/Iq8+ImjvRFaPrq4YRK9BPrXvcYxSz3DWVY98nvvPMI6gD1Nxnq8OQLPO5eH2b1n7JQ9bDzYvUBSUD05yoE9n3izPC2yOb1sjuC8wbwWvhQjMD3E6Ze9l6GdPP23bD0CFkY9LjTavBx2Cr43y0U9aUGhvFbfPr2iL5M94DEvPdiciDyeaia75NoJve//6j0C4c69QVquPDGmbjocgHI9hQnMPMQSJj0sU0C8sP+BveBqOL0i1iQ9a7h2PSxni71yvKg9BBUHvdjKUTzk+Q++6fS/PREnEb0S9xc+ejAlvge7QD2kX2c9ZedLvu0bZT1gKtu9GPBjvS662LwrRBK+HW6vvZeAjL3o5cC9FxQ9PN41FL4lkUK942NEPaEIoD2ufQM8eQEevuVqzrzNPSS9Ou7UvT2lNr3LmmO9dABhPBUSa71ZHQC9lx9yvRemt7xSL9c7FyLsvWVfoL2WW4u9BSzZO/k3J70sT5y8qDOBvTZ7dbvmd508SA9evQadnj0t00k9VTewPN5mcDxh1Bk8JO02vB8IyL0DiUm9rmFWveObfTygqRQ7HF+FPemp6z2A2Ge9wV9avSF5Aj4hwgS9SOanvAaFsD0COmo9MNhave6rATz8BkA9m+r7PMRqsryHG149MdzBPbgloDw5Do299WK2PYALnr2vxcm9zTqyPSrJmr3QJUG9WL7VveF14T1U3j48XfuRvS8GhL1Etdi93vvhu6n1ir3oddY8R+nBPH2q7D0bNNw9aY9qPTV+S73wdk69pQtUPchmfT03QT++elNwPHf0/DtgUOg6duQsvRuqOT22YKu8qkMiPYt0zD3zRdC9a4PhPNxL1bxRYbq9LMqCPfGTjTzwOtg8RrYaPP11+DslGIM9WcGKOvyPtrxSpM68jbKWPMtz5r3z0e68IE6nPRv7nL31Hyy9wWBvvXGyH726nHI9xbN3PT8cR7wHUVW88Eu7PKfD6Dv2LJ48sp33vLG3Ob3xTkY8Bn+APQQL0b0XfL+9V+YjPUNUUL5VNPk9C9xCPSPvJr7H6oi9k+7PvD3LxzxektU8nJ11PcRjBz2IL1Y9gZ4QPfHxRjxy2yQ8qJWfOf3z3z3vt3A9Fm85veofQT2lYZc917RKvQrSQ71lyE28AWZSPZAEFj1cj3I9qzsePd/BF77ilrw9Z7B7vW3UBT1zk8S8QI0iPPVo0DsQPsY7fDMBvMOMEj6eFQS93OdAPUsld73+Z+G8iphlPDBhDz7cbR29ZVloPkNx971TX788hgyqPQljqbwZiMM9OOwMvbjAc705lRS9Hx7HPS2pwL3gqQo8g2PCvb0esD1+F0Y9aB9qvAMdHT2TFeI8q40KPSb8RT0AT3i+Ns4PvmWdg73ttgO+Gb7+PNloED3uKyI9DnxMPS5WpDvaCuw8FGX2vZqmvT264dK6qT8OvV9t7L0m4UG9teskPXp19r0ZjDk9JF2CvSndET0BAD+7eOxtvOfru71gJsU75dfMveyb37zGj067aqx2PTjPMD3Uvk099HMGvelYTrxUgLs8IfT9PdKiXD3zCvm9dXrbvYK+Gb1FI5Y85xADvmxteb1HJPy8GSFWPd5Ehj1t0ao85RaMvcjwL7xOE6C9nhWpvaC3PD1M1bM9NvsYPeFS+7x5RPO7dz+iPc5RSD1yEhk9n31JPrV7mj1auzq9SWfUPN9IFb4I6vq9jA0rPe8vUr30rjW9/B1JvctMzz3Bbik9DDFHvqmjMr1T9kw8k60yPWeuLr09Ne08U3UePVJSzz3xOQY+HgKLu3/4Rz2xkk48s/MpvRHQ6TwsP+O93UdNvZ42grxt1FC9zXQDvjCu3rwHar+8teiEPGoTHj12PxS+W653vPRnuL2IkgO9INL1PLyNtLwE3Jw9mSJSPHYpiD1oI7Y9MAahPa03aL3MMB09kNQvPRcyNL2qEI479H0APjz2hD1Papa9ALvyvADv8bwO1s28TWvLvIh/j73u8iQ9RdzYO8RoEb1p6cs9MkCRvei+e73bXH29/FsMPUIl6byCIgm+2co9vOa/B766Vli7pTdEvZyw37xtwOy9XCnJPFhggbweQUI9gHXHu4S6rz0PiFC9uhiuvSuuQjvcmb49pRknPMkE3T0ZibI9llkSPQPzN71eiE89WrwHvdztiL13QpW7565rPREMHro8A+A9f/oHvcuOxb1esjc+7sg6u9DZBj4ljAY8DBVAvTGGez2+PUi9058AvtIjnT142Ia9ipmkPbDYtD0rsJs94VM3PCXjMT13ZW88hBAvPl3kn704OIa96ao4PbnRZj3Fyiw+Pt2MvTTKyzxaGU69UwreujmSmz0YaeY9IA4kvngxCj4r16W9ghmMuwSqfD1ApjI9fNusvPou6D3NEna+M8WZveF6YTxRI2O9teYYvMSTrz1qggG9z1QEO0RlFz3K8Wy8kIkrvtdrib2NJmw9ZsxNOy3Ymb0XV2u7APsVvdIyAb7L2au9xvlVPHrE6TzU1+E8ewq8vQB3x73RkBG8XZRiPTgx1byQWhE9Ck4tvH3iTD2yrzw8crenvS6zRz3HIa48AsgfPTGxPD2PjdG9NkE5vAtU8rzKiNk7NQXpvSzVv73depe97GSlPYz35j1HiQS+5XDivIjPjD0T/9283ze2veg+hz2vNmc7fPWGO9B/Nb19cP+6hNEiPofzmLv+1nC8Xggiu5fWC7ypmlS9JmxbPccjSb3nLDy9/cfUPEDLJz2Ueia+bvGSvUXy/D1dhUM8lckaPdAtT7y3kZw9f8YTPRmLB73SChc9EXw9vbfwJz6eHJs80ArZPAW0Lz1Lcwk9tPj0vUmrij0lwxS+P44rvA5pN7wK16q8OIajPTfPjL2Hk6m8HoJOvecVvrxJrpS8AIATvR51/7yQ24S9mAvQPCbVV709iV+8M0SXvZHK8j3FDza9dwh5Pejknb0ghHw9PxZlPQeUrr2UrDc9KxsAPDgOJDyC2CK9K3xhveEahTuMp0S9c83cPP4sSr0FnBm9C0nTPa129j2v8fc8a0jWvMglar2uqOC9I1MFPs0fB70dMFS+0kiNPUp1rb0oco89MFuaPPo6fDt6uoi8fAoNPd2yez1FrDq9LScrPYZsZLuDMzI998eTvE8UCb00f7w8LMq5vDZs8D35pzM937sKPI3/QbzsSBA+oPPzvR+tlTqb9iM80QTDPQ1hET1xfhg+63RFvdkT7b3e1c09G3BjO14IqL3qqyk9Wt4BPDFxNT2ZIXC8R3nUvc4nQT5vcUS+TrWGOVeembw2aY+98VdHu2xCiD0xK469ypA/vN2H+L09qDY+ORjWPYKQUb26v8o9R+f0PFd8OjvExfi9eZhCvb+WVr3W1A0+8IyivfQODLwCogo87U1cvTCAxr2f2ci8gJCkvZSlKT21YoC+TrWyvOXlDr7VLLq95GVtPXATj7ziugq9A7mLPbqM7LzQsOU74fwtvv3keDwA5KI8h+55vSBsMbzcEzO9jAqWPUL08b2qEES8pfUPPdj4iT0bjRS9quI0vVMoF70seLq9/1pkvf5Ikbzq4zg98cKaPFOWiL1mGf093LxYvV2NNTyAK609X5QXPUH05D1oYuU8qvaWuna51DxlfY88kePbvdWjsL28NEm9VEuAPTe5Bz6wOqC9MYXTPGpHRDx3dJY9irZsPU3psz2YrNq8SUoDviZwx7y/ksU81KwjvZuF+bz2bG88PP6AOwvWHz47aPG7JBcBvcUaIr3pNlQ8BzyGPekuN72NOT69m/9XveRPtj0OJEM9DAy3vQjxvTuVDSe859wdPQ7Bw7yxNto9/G+hPCCKCT4Z3Us9zw0wPTGSO73BIVS70W4mvagOaDzxWR2+KfdqvYci2zwx0767ZHy/PL5O7bzYXF48Rb6XPVYhmj3ZOSS+lE8rvXOt0b2YyvG96rVgPfYq171dO3y93rYvvEX2Xz0aiMU7saYJPqdhL77eXqU9vBOMPGvXRr1XkrY8uqJxPbJmP70K9rC9HgUHPLp/FD1k/Jw9oExRvdH+mb2NRVA9oXJuvGGKfLyjGaw9e3iRvTZHdL2wXYO8kXYfPXVyhL0Y4+y92ggePuskDb7Q1xi7VBAxvWf2Qr2Iugy+z1EcuikkAz7gflK9tTSbvYtF7Tv81fC8WxjhvXv7j7zmt8Y9EiwUvUmTfz13F509+StrO+DD5ruC9I89NV5Mvf2627k6zdi7jNnWu6DnmTxQ8349O4CuOwXwDL6q3Ye7QCQ2vWrqUj0PPQK90ccjveQ6ST2VU6w7aAf2vAjw8D1OmxK+MOGcvExCVz017w280qkHPfcpRD0NJwc+BHiGPVwOor3v7lE9cnbGPWsEkL1bnxE+jhpbvYXG9L1gxqK9z1YqPNBO9LyFS8g9MHjTvcTGgz3ra9a9HNfGvYXDK72wrRc+F4jAPMX4Ej6G/Dq+KnGxvbFFTLxopee9L8YOPpmZf7016TC9PktVvFXyFzyO+YC9ofbnveKiaryM92Q8pOUTvJbEdTwbUHi938mNPED4Kb0e1eq7TLErvU1jo7x262E89EMNvd5pjrxAvAu+ryszvf1jrrsxMI09siptvOK8R7w2wuU9noAMvVvYez0Nw5+8zkiIPbKrnz3RZLY7MAKmvdFTcb0Z7mW8rKcFvoddCr12LQq9+oESvPXg+z1NI56953/ZvWVJFT6ljrg9dVagvW5bdTka11M8auBxPYGpxb2Rbss7G8DkPNajML3bV9i8sH3MvLfMCzopcIu9FhJTPF5t2LxnhI+8EOWzPXamij1zmRS97sjzvROHMT1WNaw8+s/ZvUwvA76KApU9tL4BvXayr7yhz6E4BhyBvdY3Ij05KfA9iIFcPWMb3zw4bSK7kil2uwQNiT2bu1M9w6BBvQJ2JT1bMNu9HDnmvZtqUj0='}
reference_hashes = {}
for name, value in REFERENCE_FILES.items():
    content = base64.b64decode(value)
    (REFERENCE/name).write_bytes(content)
    reference_hashes[name] = hashlib.sha256(content).hexdigest()
enrollment_candidates = (list(Path('/kaggle/input').rglob('auditor_enrollment.wav'))
                         if ON_KAGGLE else ['/tmp/battousai-reference-v1/auditor_enrollment.wav'])
enrollment_candidates = [Path(path) for path in enrollment_candidates if Path(path).is_file()]
if not enrollment_candidates:
    raise RuntimeError('The attached Kaggle dataset must contain auditor_enrollment.wav')
enrollment_hashes = {hashlib.sha256(path.read_bytes()).hexdigest(): path
                     for path in enrollment_candidates}
if len(enrollment_hashes) > 1:
    raise RuntimeError('Found multiple different auditor_enrollment.wav files in attached datasets')
enrollment_source = next(iter(enrollment_hashes.values()))
shutil.copy2(enrollment_source, REFERENCE/'auditor_enrollment.wav')
reference_hashes['auditor_enrollment.wav'] = next(iter(enrollment_hashes))
(RESULTS/'reference-hashes.json').write_text(json.dumps(reference_hashes, indent=2))
print('Reference bundle ready:', reference_hashes)


## Credentials, checkpoint restore, and attached video

Create a private Kaggle secret named `HF_TOKEN`. The token is read from the environment and is never embedded or printed. The current video is copied from the attached dataset because YouTube blocks Kaggle's shared addresses.


In [ ]:
if not ON_KAGGLE:
    import getpass
    ENV['HF_TOKEN'] = os.environ.get('HF_TOKEN') or os.environ.get('HUGGINGFACE_TOKEN') or getpass.getpass('Hugging Face token: ')
if not ENV['HF_TOKEN']:
    raise RuntimeError('A Hugging Face token with diarization-model access is required.')
ENV['HUGGING_FACE_HUB_TOKEN'] = ENV['HF_TOKEN']
if ON_KAGGLE:
    for archive in Path('/kaggle/input').rglob('stage-checkpoints*.zip'):
        with zipfile.ZipFile(archive) as zipped:
            for member in zipped.infolist():
                target = (BASE/member.filename).resolve()
                if not target.is_relative_to(CACHE.resolve()):
                    raise RuntimeError('Unexpected checkpoint archive path')
            zipped.extractall(BASE)
    mossformer_cache = CACHE/'mossformer2-items'
    for archive in Path('/kaggle/input').rglob('mossformer2-checkpoints*.zip'):
        with zipfile.ZipFile(archive) as zipped:
            for member in zipped.infolist():
                target = (mossformer_cache/member.filename).resolve()
                if not target.is_relative_to(mossformer_cache.resolve()):
                    raise RuntimeError('Unexpected MossFormer2 checkpoint archive path')
            zipped.extractall(mossformer_cache)
        print('Restored per-window MossFormer2 checkpoints:', archive)
    overlap_cache = CACHE/'overlap-extraction-items'
    for archive in Path('/kaggle/input').rglob('overlap-extraction-checkpoints*.zip'):
        with zipfile.ZipFile(archive) as zipped:
            for member in zipped.infolist():
                target = (overlap_cache/member.filename).resolve()
                if not target.is_relative_to(overlap_cache.resolve()):
                    raise RuntimeError('Unexpected overlap checkpoint archive path')
            zipped.extractall(overlap_cache)
        print('Restored per-window overlap checkpoints:', archive)
    expanded_checkpoints = [path for path in Path('/kaggle/input').rglob(VIDEO_ID)
                            if path.is_dir() and path.parent.name == 'stage-cache']
    if len(expanded_checkpoints) > 1:
        raise RuntimeError('Found more than one expanded checkpoint dataset for this video')
    if expanded_checkpoints:
        shutil.copytree(expanded_checkpoints[0], CACHE, dirs_exist_ok=True)
        print('Restored expanded stage checkpoints:', expanded_checkpoints[0])
OVERLAP_POLICY = None
policy_matches = []
search_root = Path('/kaggle/input') if ON_KAGGLE else Path.cwd()
for head_path in search_root.rglob('head-to-head.json'):
    try:
        head = json.loads(head_path.read_text())
    except Exception:
        continue
    candidate = head_path.parent/'overlap-review-policy.json'
    if (head.get('video_id') == VIDEO_ID
            and head.get('revision') == 'sortformer-additive-two-tier-policy-v5'
            and candidate.is_file()):
        policy_matches.append(candidate)
# Kaggle normally expands dataset archives, but accept a retained ZIP too.
for archive in search_root.rglob('*.zip'):
    try:
        with zipfile.ZipFile(archive) as zipped:
            names = set(zipped.namelist())
            for head_name in [name for name in names if name.endswith('head-to-head.json')]:
                head = json.loads(zipped.read(head_name))
                policy_name = str(Path(head_name).parent/'overlap-review-policy.json')
                if (head.get('video_id') == VIDEO_ID
                        and head.get('revision') == 'sortformer-additive-two-tier-policy-v5'
                        and policy_name in names):
                    extracted = WORK/'attached-overlap-review-policy.json'
                    extracted.write_bytes(zipped.read(policy_name))
                    policy_matches.append(extracted)
    except (zipfile.BadZipFile, KeyError, json.JSONDecodeError):
        continue
unique_policies = []
for candidate in policy_matches:
    if not any(candidate.read_bytes() == existing.read_bytes() for existing in unique_policies):
        unique_policies.append(candidate)
if len(unique_policies) == 1:
    OVERLAP_POLICY = unique_policies[0]
    print('Found required additive Sortformer policy:', OVERLAP_POLICY)
elif len(unique_policies) > 1:
    raise RuntimeError('Found multiple different matching v5 overlap policies')
elif REQUIRE_OVERLAP_POLICY:
    raise RuntimeError(
        'Attach the Kaggle dataset created from sortformer-comparison-results(4).zip. '
        'No matching v5 overlap policy was found; stopping before the full run.')
if not VIDEO.exists():
    if ON_KAGGLE:
        accepted_names = {VIDEO_ID+'.mp4', VIDEO_ID+'_full480.mp4'}
        matches = [path for path in Path('/kaggle/input').rglob('*.mp4')
                   if path.name in accepted_names]
        if not matches:
            # Private datasets created for a single holdout commonly use the
            # generic name video.mp4. Accept it only when it is the sole MP4
            # across all attached inputs, so selection remains unambiguous.
            all_mp4 = list(Path('/kaggle/input').rglob('*.mp4'))
            if len(all_mp4) == 1:
                matches = all_mp4
        if len(matches) != 1:
            raise RuntimeError(
                'Attach exactly one video: an ID-named file from '
                f'{sorted(accepted_names)}, or a sole video.mp4 across all inputs. '
                'YouTube blocks downloads from Kaggle.')
        source_video = matches[0]
    else:
        local_video = Path.cwd()/(VIDEO_ID+'.mp4')
        if not local_video.is_file():
            raise RuntimeError(f'Missing local video: {{local_video}}')
        source_video = local_video
    # The supplied YouTube video uses AV1, which Kaggle's OpenCV build cannot
    # decode. Normalize it once so the full visual pass actually reads frames.
    checked(['ffmpeg', '-nostdin', '-hide_banner', '-loglevel', 'error', '-y',
             '-i', str(source_video), '-map', '0:v:0', '-map', '0:a:0',
             '-c:v', 'libx264', '-preset', 'fast', '-crf', '22',
             '-pix_fmt', 'yuv420p', '-c:a', 'aac', '-b:a', '160k', str(VIDEO)])
video_codec = subprocess.check_output(
    ['ffprobe', '-v', 'error', '-select_streams', 'v:0',
     '-show_entries', 'stream=codec_name', '-of', 'default=nw=1:nk=1', str(VIDEO)],
    env=ENV, text=True).strip()
if video_codec != 'h264':
    raise RuntimeError(f'Expected normalized H.264 video, found {{video_codec!r}}')
print('Normalized video codec:', video_codec)
print('Credentials configured; token not displayed.')
print('Video ready:', VIDEO, VIDEO.stat().st_size, 'bytes')
(RESULTS/'run-input.json').write_text(json.dumps({
    'video_url': VIDEO_URL,
    'video_id': VIDEO_ID,
    'normalized_video_sha256': hashlib.sha256(VIDEO.read_bytes()).hexdigest(),
    'notebook_revision': NOTEBOOK_REVISION,
}, indent=2))


In [ ]:
def export_checkpoints():
    if CACHE.exists():
        shutil.make_archive(str(BASE/'stage-checkpoints'), 'zip', BASE,
                            str(CACHE.relative_to(BASE)))

def stream(command, log_name, failure):
    log_path = RESULTS/log_name
    recent_lines = []
    with log_path.open('w') as log:
        process = subprocess.Popen(command, cwd=WORK, env=ENV, stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in process.stdout:
            line = line.replace(ENV['HF_TOKEN'], '[REDACTED]')
            log.write(line); log.flush()
            recent_lines.append(line.rstrip())
            recent_lines = recent_lines[-20:]
            print(line if len(line) < 1000 else line[:1000]+' ... [full line saved]\n', end='')
        if process.wait() != 0:
            tail = '\n'.join(recent_lines)
            raise RuntimeError(f"{failure}\nLog: {log_path}\nLast output:\n{tail}")

def run_test(video, stem, batch_size=4):
    output = RESULTS/(stem+'_evidence.json')
    command = [PYTHON, str(WORK/'chainofrules.py'), str(video),
        '--voice-priors', str(REFERENCE/'voice_embeddings.npy'),
        '--face-priors', str(REFERENCE/'face_embeddings.npy'),
        '--output', str(output), '--cache-dir', str(CACHE), '--batch-size', str(batch_size)]
    started = time.monotonic()
    try:
        stream(command, stem+'.log', 'Pipeline failed; see the saved log. Retry with BATCH_SIZE=1 for CUDA memory errors.')
    finally:
        export_checkpoints()
    result = json.loads(output.read_text())
    text_repeat_candidates = result.get('text_repeat_candidates', [])
    (RESULTS/(stem+'_text_repeat_candidates.json')).write_text(
        json.dumps(text_repeat_candidates, indent=2)+'\n')
    text_repeat_rows = [
        f"[{row['left_start']:.2f}-{row['left_end']:.2f}] {row['left_text']}  <=>  "
        f"[{row['right_start']:.2f}-{row['right_end']:.2f}] {row['right_text']}  "
        f"(similarity={row['text_similarity']:.3f})"
        for row in text_repeat_candidates
    ]
    (RESULTS/(stem+'_text_repeat_candidates.txt')).write_text(
        '\n'.join(text_repeat_rows)+'\n')
    transcript = '\n'.join(f"[{s['start']:.2f}-{s['end']:.2f}] {s['final_speaker']} (strength={s['final_confidence']:.2f}): {s['text']}" for s in result['segments'])
    (RESULTS/(stem+'_transcript.txt')).write_text(transcript+'\n')
    repeat_rows = []
    for segment in result['segments']:
        for evidence in segment.get('evidence', []):
            if evidence.get('source') == 'repeated_presentation':
                d = evidence.get('details', {})
                repeat_rows.append(f"[{segment['start']:.2f}] {segment['text']}  <=>  [{d.get('donor_start', 0):.2f}] {d.get('donor_text', '')}")
    (RESULTS/(stem+'_repeat_candidates.txt')).write_text('\n'.join(repeat_rows)+'\n')
    print(f'Elapsed: {(time.monotonic()-started)/60:.1f} minutes')
    print('Repeated-presentation groups:', len(result.get('repeated_presentations', [])))
    print('Short text-repeat review candidates:', len(text_repeat_candidates))
    return result

def extract_clip(start, duration, name):
    clip = WORK/name
    checked(['ffmpeg', '-nostdin', '-hide_banner', '-loglevel', 'error', '-y',
             '-ss', str(start), '-i', str(VIDEO), '-t', str(duration),
             '-c:v', 'libx264', '-preset', 'fast', '-crf', '20', '-c:a', 'aac', str(clip)])
    return clip

def run_targeted_review():
    output_dir = RESULTS/'targeted-review'
    command = [PYTHON, str(WORK/'review_transcript_regions.py'), '--video', str(VIDEO),
        '--baseline', str(RESULTS/'full_video_evidence.json'),
        '--target-reference', str(REFERENCE/'voice_embeddings.npy'),
        '--output-dir', str(output_dir), '--cache-dir', str(CACHE/'targeted-review'),
        '--weak-confidence', str(REVIEW_WEAK_CONFIDENCE), '--short-seconds', str(REVIEW_SHORT_SECONDS),
        '--minimum-gap', '5', '--context-seconds', '3', '--maximum-window-seconds', '30',
        '--window-overlap-seconds', '4', '--device', 'cuda' if ON_KAGGLE else 'auto']
    if HAS_OPENING_REFERENCE:
        command += ['--other-reference',
                    'Opening_officer='+str(REFERENCE/'opening_officer_reference.npy')]
    before = (RESULTS/'full_video_evidence.json').read_bytes()
    try:
        stream(command, 'targeted-review.log', 'Targeted review failed; see the saved log.')
    finally:
        export_checkpoints()
    assert (RESULTS/'full_video_evidence.json').read_bytes() == before
    return json.loads((output_dir/'comparison_summary.json').read_text())

def run_overlap_extraction():
    output_dir = RESULTS/'overlap-extraction'
    completed_report = output_dir/'report.json'
    if completed_report.is_file():
        saved = json.loads(completed_report.read_text())
        summary = saved.get('summary', {})
        if summary.get('selected') == summary.get('completed'):
            print('Reusing completed overlap extraction:', completed_report)
            return saved
    command = [PYTHON, str(WORK/'review_overlap_extraction.py'), '--video', str(VIDEO),
        '--baseline', str(RESULTS/'full_video_evidence.json'),
        '--enrollment', str(REFERENCE/'auditor_enrollment.wav'),
        '--voice-priors', str(REFERENCE/'voice_embeddings.npy'),
        '--output-dir', str(output_dir), '--device', 'cuda' if ON_KAGGLE else 'cpu',
        '--whisper-model', 'large-v2',
        '--cache-dir', str(CACHE/'overlap-extraction-items'),
        '--snapshot-archive', str(BASE/'overlap-extraction-checkpoints.zip'),
        '--snapshot-every', '5']
    if OVERLAP_POLICY is not None:
        command += ['--selection-policy', str(OVERLAP_POLICY)]
        print('Using additive Sortformer policy:', OVERLAP_POLICY)
    before = (RESULTS/'full_video_evidence.json').read_bytes()
    try:
        stream(command, 'overlap-extraction.log', 'Overlap extraction failed; baseline results remain valid.')
    finally:
        export_checkpoints()
    assert (RESULTS/'full_video_evidence.json').read_bytes() == before
    return json.loads((output_dir/'report.json').read_text())

def run_mossformer2_review():
    """Separate overlap candidates without modifying baseline text or identity."""
    output_dir = RESULTS/'mossformer2-review'
    inference_dir = output_dir/'inference'
    labels = output_dir/'selected-overlaps.json'
    output_dir.mkdir(parents=True, exist_ok=True)
    prepare = [PYTHON, str(WORK/'mossformer2_review_policy.py'), 'prepare',
        '--baseline', str(RESULTS/'full_video_evidence.json'), '--output', str(labels)]
    if OVERLAP_POLICY is not None:
        prepare += ['--selection-policy', str(OVERLAP_POLICY)]
    checked(prepare, cwd=WORK)
    command = [PYTHON, '-B', str(WORK/'run_mossformer2_separation_experiment.py'),
        '--video', str(VIDEO), '--labels', str(labels),
        '--voice-priors', str(REFERENCE/'voice_embeddings.npy'),
        '--output-dir', str(inference_dir), '--context', '3',
        '--device', 'cuda' if ON_KAGGLE else 'cpu',
        '--hf-home', str(BASE/'huggingface-cache'),
        '--cache-dir', str(CACHE/'mossformer2-items'),
        '--snapshot-archive', str(BASE/'mossformer2-checkpoints.zip'),
        '--snapshot-every', '5']
    ENV['SPEECHBRAIN_CACHE'] = str(
        BASE/'speechbrain-cache'/'spkrec-ecapa-voxceleb')
    before = (RESULTS/'full_video_evidence.json').read_bytes()
    try:
        stream(command, 'mossformer2-review.log',
               'MossFormer2 review failed; baseline results remain valid.')
    finally:
        export_checkpoints()
    report = output_dir/'review-evidence.json'
    checked([PYTHON, str(WORK/'mossformer2_review_policy.py'), 'evaluate',
             '--report', str(inference_dir/'report.json'), '--output', str(report)],
            cwd=WORK)
    assert (RESULTS/'full_video_evidence.json').read_bytes() == before
    return json.loads(report.read_text())

def run_caption_gap_review():
    """Use optional YouTube timing evidence to find review-only transcript gaps."""
    output_dir = RESULTS/'caption-gap-review'
    status_path = RESULTS/'caption-gap-status.json'
    caption_dir = WORK/'youtube-captions'; caption_dir.mkdir(exist_ok=True)
    attached_roots = [Path('/kaggle/input')] if ON_KAGGLE else []
    captions = []
    for root in attached_roots:
        captions.extend(root.rglob(VIDEO_ID+'.en-orig.json3'))
        captions.extend(root.rglob(VIDEO_ID+'.en.json3'))
    output_template = caption_dir/(VIDEO_ID+'.%(ext)s')
    if not captions:
        command = [PYTHON, '-m', 'yt_dlp', '--skip-download', '--write-auto-subs',
            '--sub-langs', 'en-orig,en', '--sub-format', 'json3',
            '-o', str(output_template), VIDEO_URL]
        download = subprocess.run(command, cwd=WORK, env=ENV, text=True,
                                  stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
        (RESULTS/'caption-download.log').write_text(download.stdout)
        captions = sorted(caption_dir.glob(VIDEO_ID+'.en-orig.json3'))
        if not captions:
            captions = sorted(caption_dir.glob(VIDEO_ID+'.en.json3'))
    if not captions:
        status = {
            'status': 'captions_unavailable',
            'review_candidates': 0,
            'automatic_text_insertion': False,
            'speaker_identity_changed': False,
        }
        status_path.write_text(json.dumps(status, indent=2)+'\n')
        print('Caption gap review skipped: automatic English captions unavailable.')
        return status
    if output_dir.exists():
        shutil.rmtree(output_dir)
    before = (RESULTS/'full_video_evidence.json').read_bytes()
    checked([PYTHON, str(WORK/'build_caption_gap_review.py'),
             '--captions', str(captions[0]),
             '--baseline', str(RESULTS/'full_video_evidence.json'),
             '--video', str(VIDEO), '--output-dir', str(output_dir)], cwd=WORK)
    manifest = json.loads((output_dir/'manifest.json').read_text())
    status = {
        'status': 'review_ready',
        'caption_type': 'youtube_automatic',
        'review_candidates': len(manifest),
        'nearby_transcript_duplicates_suppressed': True,
        'captions_do_not_identify_speakers': True,
        'automatic_text_insertion': False,
        'speaker_identity_changed': False,
    }
    status_path.write_text(json.dumps(status, indent=2)+'\n')
    assert (RESULTS/'full_video_evidence.json').read_bytes() == before
    return status

def run_diaper_overlap():
    """Run official DiaPer on full audio, then compare without changing baseline."""
    source = WORK/'vendor'/'DiaPer'
    checkpoint = source/'models'/'10attractors'/'SC_LibriSpeech_2spk_adapted1-10'/'models'/'checkpoint_100.tar'
    infer_config = source/'examples'/'infer_16k_10attractors.yaml'
    if not checkpoint.is_file():
        if source.exists():
            shutil.rmtree(source)
        source.parent.mkdir(parents=True, exist_ok=True)
        checked(['git', 'clone', '--depth', '1', '--filter=blob:none', '--no-checkout',
                 'https://github.com/BUTSpeechFIT/DiaPer.git', str(source)])
        checked(['git', '-C', str(source), 'sparse-checkout', 'init', '--no-cone'])
        checked(['git', '-C', str(source), 'sparse-checkout', 'set',
                 '/diaper/', '/examples/infer_16k_10attractors.yaml',
                 '/models/10attractors/SC_LibriSpeech_2spk_adapted1-10/models/checkpoint_100.tar'])
        checked(['git', '-C', str(source), 'checkout'])
    # DiaPer relies on a small Perceiver change from the authors' Transformers
    # fork. Install it into a private overlay so the established pipeline keeps
    # its own dependency set.
    transformer_overlay = source/'python-overlay'
    if not (transformer_overlay/'transformers').is_dir():
        checked([PYTHON, '-m', 'pip', 'install', '--no-deps', '--target',
                 str(transformer_overlay),
                 'git+https://github.com/fnlandini/transformers.git@b830ec2245139b157576153cfd8999e1da24a82c'])
    # DiaPer imports only the Perceiver model and does not use tokenization.
    # Keep the host pipeline's current tokenizers build and disable only this
    # irrelevant upper-bound check inside DiaPer's private overlay.
    dependency_check = transformer_overlay/'transformers'/'dependency_versions_check.py'
    dependency_text = dependency_check.read_text()
    runtime_loop = 'for pkg in pkgs_to_check_at_runtime:\n'
    skip_marker = '    if pkg == "tokenizers":  # unused by DiaPer\n        continue\n'
    if skip_marker not in dependency_text:
        if runtime_loop not in dependency_text:
            raise RuntimeError('Could not patch DiaPer Transformers dependency checks')
        dependency_text = dependency_text.replace(
            runtime_loop, runtime_loop + skip_marker, 1)
    dependency_check.write_text(dependency_text)
    # The official 2023 script's GPU check treats GPU index 0 as CPU and asks
    # safe_gpu to allocate devices. Kaggle already assigned CUDA_VISIBLE_DEVICES,
    # so use that allocation directly.
    infer_script = source/'diaper'/'infer_single_file.py'
    infer_text = infer_script.read_text()
    infer_text = infer_text.replace(
        "if args.gpu >= 1:",
        "if args.gpu >= 0 and torch.cuda.is_available():")
    infer_text = infer_text.replace(
        "        safe_gpu.claim_gpus(nb_gpus=args.gpu)\n", "")
    infer_text = infer_text.replace(
        "librosa.get_duration(filename=filepath)", "sf.info(filepath).duration")
    infer_script.write_text(infer_text)
    models_script = source/'diaper'/'backend'/'models.py'
    models_text = models_script.read_text().replace(
        "map_location=args.device)", "map_location=args.device, weights_only=False)").replace(
        "map_location=device)", "map_location=device, weights_only=False)")
    models_script.write_text(models_text)
    # Librosa 0.10+ made mel-filter arguments keyword-only. Retain DiaPer's
    # published feature settings while adapting the call syntax.
    features_script = source/'diaper'/'common_utils'/'features.py'
    features_text = features_script.read_text()
    legacy_mel_call = 'librosa.filters.mel(sampling_rate, n_fft, feature_dim)'
    current_mel_call = 'librosa.filters.mel(sr=sampling_rate, n_fft=n_fft, n_mels=feature_dim)'
    if legacy_mel_call in features_text:
        features_text = features_text.replace(legacy_mel_call, current_mel_call)
    if current_mel_call not in features_text:
        raise RuntimeError('Could not patch DiaPer for the current Librosa API')
    features_script.write_text(features_text)

    audio_dir = RESULTS/'diaper-overlap'/'input'
    audio_dir.mkdir(parents=True, exist_ok=True)
    audio = audio_dir/'full-video.wav'
    if not audio.is_file():
        checked(['ffmpeg', '-nostdin', '-hide_banner', '-loglevel', 'error', '-y',
                 '-i', str(VIDEO), '-vn', '-ac', '1', '-ar', '16000', str(audio)])
    output_dir = RESULTS/'diaper-overlap'/'inference'
    command = [PYTHON, str(infer_script), '-c', str(infer_config),
        '--wav-dir', str(audio_dir), '--wav-name', 'full-video',
        '--models-path', str(checkpoint.parent), '--epochs', '100',
        '--rttms-dir', str(output_dir), '--gpu', '0']
    prior_pythonpath = ENV.get('PYTHONPATH', '')
    ENV['PYTHONPATH'] = str(transformer_overlay) + os.pathsep + prior_pythonpath
    try:
        stream(command, 'diaper-overlap.log',
               'DiaPer inference failed; baseline and existing overlap results remain valid.')
    finally:
        ENV['PYTHONPATH'] = prior_pythonpath
    rttms = list(output_dir.rglob('full-video.rttm'))
    if len(rttms) != 1:
        raise RuntimeError(f'Expected one DiaPer RTTM, found {{len(rttms)}}')
    report_path = RESULTS/'diaper-overlap'/'comparison.json'
    checked([PYTHON, str(WORK/'evaluate_diaper_overlap.py'),
             '--baseline', str(RESULTS/'full_video_evidence.json'),
             '--rttm', str(rttms[0]), '--output', str(report_path)], cwd=WORK)
    return json.loads(report_path.read_text())

def export_reference_promotion_review():
    output_dir = RESULTS/'reference-promotion-review'
    manifest_path = output_dir/'manifest.json'
    if manifest_path.is_file():
        print('Reusing completed reference-promotion review:', manifest_path)
        return json.loads(manifest_path.read_text())
    if output_dir.exists():
        print('Removing incomplete reference-promotion review:', output_dir)
        shutil.rmtree(output_dir)
    command = [PYTHON, str(WORK/'reference_promotion.py'), 'export',
        '--video', str(VIDEO), '--evidence', str(RESULTS/'full_video_evidence.json'),
        '--output-dir', str(output_dir), '--source-url', VIDEO_URL,
        '--reference-metadata', str(REFERENCE/'reference.json')]
    checked(command, cwd=WORK)
    return json.loads(manifest_path.read_text())


## Run the opening check, whole video, and additive reviews

The opening check confirms that face analysis is actually using CUDA. The established stages run first. Automatic captions, when available, identify possible transcript gaps and produce short review clips after nearby wording duplicates are suppressed. Captions never identify speakers or insert text. When the matching v5 Sortformer result dataset is attached, speaker-conditioned extraction processes the union of existing baseline overlap intervals and strong Sortformer additions. MossFormer2 then separates those same review candidates, rejects weak target matches, and exports review-only evidence; it never inserts text or changes speaker identity. DiaPer remains a read-only comparison.


In [ ]:
opening_clip = extract_clip(0, 30, 'opening_30s.mp4')
opening = run_test(opening_clip, 'opening', BATCH_SIZE)
providers = opening.get('runtime', {}).get('face_providers', {})
if ON_KAGGLE and (not providers or any('CUDAExecutionProvider' not in p for p in providers.values())):
    raise RuntimeError('Face models are still on CPU. Review opening.log before starting the full video.')
print((RESULTS/'opening_transcript.txt').read_text())

if RUN_FULL_VIDEO:
    full_video = run_test(VIDEO, 'full_video', BATCH_SIZE)
    decoded_visual_segments = sum(
        1 for segment in full_video['segments']
        for evidence in segment.get('evidence', [])
        if evidence.get('source') == 'visual_context'
        and evidence.get('details', {}).get('frames_read', 0) > 0)
    if decoded_visual_segments == 0:
        raise RuntimeError('No full-video frames were decoded; supplemental reviews were not started.')
    print('Full-video segments with decoded visual frames:', decoded_visual_segments)
    confident_dir = RESULTS/'confidence-filtered-transcript'
    checked([PYTHON, str(WORK/'export_confident_transcript.py'),
             str(RESULTS/'full_video_evidence.json'),
             '--output-dir', str(confident_dir),
             '--target-minimum', '0.35', '--other-minimum', '0.65'], cwd=WORK)
    print('Confidence-filtered transcript:', confident_dir/'confident_transcript.txt')
    if RUN_CAPTION_GAP_REVIEW:
        caption_gap_review = run_caption_gap_review()
        print('Caption gap review:', json.dumps(caption_gap_review, indent=2))
    if RUN_TARGETED_REVIEW:
        targeted_review = run_targeted_review()
        print('Targeted review:', json.dumps(targeted_review, indent=2))
    if RUN_OVERLAP_EXTRACTION:
        overlap_review = run_overlap_extraction()
        print('Overlap extraction:', json.dumps(overlap_review['summary'], indent=2))
    if RUN_MOSSFORMER2_REVIEW:
        mossformer2_review = run_mossformer2_review()
        print('MossFormer2 review evidence:',
              json.dumps(mossformer2_review['summary'], indent=2))
    if RUN_DIAPER_OVERLAP:
        diaper_review = run_diaper_overlap()
        print('DiaPer overlap comparison:', json.dumps(diaper_review['summary'], indent=2))
    promotion_review = export_reference_promotion_review()
    print('Reference promotion candidates:', len(promotion_review['candidates']))
else:
    print('Full video disabled. Review the opening result, then set RUN_FULL_VIDEO=True.')


## Download results

`diarization-results.zip` contains the preserved baseline, caption-gap review clips when captions are available, existing supplemental reviews, MossFormer2 review-only evidence, DiaPer RTTM and comparison report, logs, and package versions. `stage-checkpoints.zip` restarts the complete workflow. `overlap-extraction-checkpoints.zip` and `mossformer2-checkpoints.zip` are refreshed every five windows and can be attached directly if Kaggle stops during either long stage.


In [ ]:
from IPython.display import FileLink, display
export_checkpoints()
with (RESULTS/'runtime-packages.txt').open('w') as packages:
    checked([PYTHON, '-m', 'pip', 'freeze'], stdout=packages)
result_zip = Path(shutil.make_archive(str(BASE/'diarization-results'), 'zip', RESULTS))
checkpoint_zip = BASE/'stage-checkpoints.zip'
mossformer2_zip = BASE/'mossformer2-checkpoints.zip'
overlap_zip = BASE/'overlap-extraction-checkpoints.zip'
prior_cwd = Path.cwd()
try:
    os.chdir(BASE)
    display(FileLink(result_zip.name))
    if checkpoint_zip.is_file():
        display(FileLink(checkpoint_zip.name))
    if mossformer2_zip.is_file():
        display(FileLink(mossformer2_zip.name))
    if overlap_zip.is_file():
        display(FileLink(overlap_zip.name))
finally:
    os.chdir(prior_cwd)
if ON_KAGGLE:
    # Kaggle publishes everything under /kaggle/working. Keep only the four
    # downloadable archives instead of uploading the virtual environment,
    # model caches, source video, and unpacked duplicate results.
    cleanup_paths = [
        RESULTS, CACHE.parent, WORK, VENV, BASE/'bootstrap-tools',
        BASE/'huggingface-cache', BASE/'speechbrain-cache',
        BASE/'matplotlib-cache', BASE/'numba-cache', BASE/'insightface',
    ]
    for cleanup_path in cleanup_paths:
        if cleanup_path.is_dir():
            shutil.rmtree(cleanup_path, ignore_errors=True)
        elif cleanup_path.exists():
            cleanup_path.unlink()
print('Saved in:', BASE)
print('If Kaggle blocks a link, download the same ZIP from the Output panel.')
